<a href="https://colab.research.google.com/github/rxphaelbihag/Data-Analysis-with-R/blob/main/Multinomial_Logit_Modeling_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multinomial Logit Model Sample

## Introduction
Goal: Predicting DietType (Mostly Plant-Based, Balanced, Mostly Animal-Based)
Predictors:
-


## Data Wrangling

### Importing Dataset from Kaggle and Necessary Libraries

In [31]:
# Kaggle credentials Set Up Directly into the code
Sys.setenv(KAGGLE_USERNAME = "raphaelkhandiebihag")
Sys.setenv(KAGGLE_KEY = "KGAT_71b38028c84a05aab1dfb918cd8e716b")

# Install the Kaggle CLI tool
system("pip install kaggle --upgrade --quiet", wait = TRUE)

# Download the dataset
system("kaggle datasets download naveennas/sustainable-lifestyle-rating-dataset", wait = TRUE)

# Unzip the downloaded file
system("unzip sustainable-lifestyle-rating-dataset.zip -d dataset", wait = TRUE)

In [44]:
# Import the necessary packages
library(tidyverse)
library(nnet)

In [33]:
# Import the dataset
sustainability_data <- read_csv("/content/dataset/lifestyle_sustainability_data.csv")
head(sustainability_data)

Rows: 499 Columns: 20
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (12): Location, DietType, LocalFoodFrequency, TransportationMode, Energy...
dbl  (7): ParticipantID, Age, HomeSize, EnvironmentalAwareness, MonthlyElect...
lgl  (1): SustainableBrands

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


ParticipantID,Age,Location,DietType,LocalFoodFrequency,TransportationMode,EnergySource,HomeType,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,Gender,UsingPlasticProducts,DisposalMethods,PhysicalActivities,Rating
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<lgl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,35,Urban,Mostly Plant-Based,Often,Bike,Renewable,Apartment,800,Rarely,TRUE,5,High,100,1500,Female,Rarely,Composting,High,5
2,28,Suburban,Balanced,Sometimes,Public Transit,Mixed,House,1500,Sometimes,TRUE,4,Moderate,250,3000,Male,Sometimes,Recycling,Moderate,4
3,65,Rural,Mostly Animal-Based,Rarely,Car,Non-Renewable,House,2500,Often,FALSE,2,Low,400,4500,Male,Often,Landfill,Low,1
4,42,Urban,Mostly Plant-Based,Often,Walk,Renewable,Apartment,950,Sometimes,TRUE,4,Moderate,150,2000,Female,Rarely,Recycling,High,5
5,31,Suburban,Balanced,Sometimes,Public Transit,Mixed,House,1800,Often,TRUE,3,Low,300,3500,Non-Binary,Sometimes,Combination,Moderate,3
6,58,Rural,Mostly Animal-Based,Rarely,Car,Non-Renewable,House,2200,Always,FALSE,1,None,450,5000,Male,Often,Landfill,None,1


In [34]:
# Creating a subset
new_sustainability_data <- subset(sustainability_data, select = c(DietType, Age, Gender, Rating, EnvironmentalAwareness, SustainableBrands, LocalFoodFrequency, DisposalMethods))
head(new_sustainability_data)

DietType,Age,Gender,Rating,EnvironmentalAwareness,SustainableBrands,LocalFoodFrequency,DisposalMethods
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
Mostly Plant-Based,35,Female,5,5,TRUE,Often,Composting
Balanced,28,Male,4,4,TRUE,Sometimes,Recycling
Mostly Animal-Based,65,Male,1,2,FALSE,Rarely,Landfill
Mostly Plant-Based,42,Female,5,4,TRUE,Often,Recycling
Balanced,31,Non-Binary,3,3,TRUE,Sometimes,Combination
Mostly Animal-Based,58,Male,1,1,FALSE,Rarely,Landfill


In [35]:
summary(new_sustainability_data)

      DietType        Age              Gender        Rating     
 Length   :499   Min.   :18.00   Length   :499   Min.   :1.000  
 N.unique :  3   1st Qu.:31.00   N.unique :  4   1st Qu.:2.000  
 N.blank  :  0   Median :44.00   N.blank  :  0   Median :4.000  
 Min.nchar:  8   Mean   :44.05   Min.nchar:  4   Mean   :3.431  
 Max.nchar: 19   3rd Qu.:58.00   Max.nchar: 17   3rd Qu.:5.000  
                 Max.   :96.00                   Max.   :5.000  
 EnvironmentalAwareness SustainableBrands LocalFoodFrequency  DisposalMethods
 Min.   :1.000          Mode :logical     Length   :499      Length   :499   
 1st Qu.:2.000          FALSE:236         N.unique :  4      N.unique :  4   
 Median :3.000          TRUE :263         N.blank  :  0      N.blank  :  0   
 Mean   :3.062                            Min.nchar:  5      Min.nchar:  8   
 3rd Qu.:4.000                            Max.nchar:  9      Max.nchar: 11   
 Max.   :5.000                                                               

### Proper Encoding and Releveling
There seems to be 4 unique values for gender. We have to get rid of the "Prefer not to say" option.

Predictors such as Gender, SustainableBrands, LocalFoodFrequency, and DisposalMethods are in different data types. We'll have to turn them into factors.

After that, we have to set the baseline for the variables (releveling).



In [36]:
# Inspect the unique values of certain columns
gender_uniq <- unique(new_sustainability_data$Gender)
localfood_uniq <- unique(new_sustainability_data$LocalFoodFrequency)
disposal_uniq <- unique(new_sustainability_data$DisposalMethods)

gender_uniq
localfood_uniq
disposal_uniq

[1] "Female"            "Male"              "Non-Binary"       
[4] "Prefer not to say"

[1] "Often"     "Sometimes" "Rarely"    "Always"

[1] "Composting"  "Recycling"   "Landfill"    "Combination"

In [40]:
# Keeps rows where Gender is NOT "Prefer not to say"
new_sustainability_data <- new_sustainability_data[new_sustainability_data$Gender != "Prefer not to say", ]
summary(new_sustainability_data)

      DietType        Age            Gender        Rating     
 Length   :424   Min.   :18   Female    :167   Min.   :1.000  
 N.unique :  3   1st Qu.:31   Male      :176   1st Qu.:2.000  
 N.blank  :  0   Median :44   Non-Binary: 81   Median :4.000  
 Min.nchar:  8   Mean   :44                    Mean   :3.413  
 Max.nchar: 19   3rd Qu.:58                    3rd Qu.:5.000  
                 Max.   :96                    Max.   :5.000  
 EnvironmentalAwareness SustainableBrands LocalFoodFrequency    DisposalMethods
 Min.   :1.000          FALSE:197         Always   : 66      Combination: 73   
 1st Qu.:2.000          TRUE :227         Often    :124      Composting :102   
 Median :3.000                            Rarely   :112      Landfill   :124   
 Mean   :3.092                            Sometimes:122      Recycling  :125   
 3rd Qu.:4.000                                                                 
 Max.   :5.000                                                                 

In [41]:
# Data Encoding: Converting the columns as factors
new_sustainability_data$DietType <- as.factor(new_sustainability_data$DietType)
new_sustainability_data$Gender <- as.factor(new_sustainability_data$Gender)
new_sustainability_data$SustainableBrands <- as.factor(new_sustainability_data$SustainableBrands)
new_sustainability_data$LocalFoodFrequency <- as.factor(new_sustainability_data$LocalFoodFrequency)
new_sustainability_data$DisposalMethods <- as.factor(new_sustainability_data$DisposalMethods)

head(new_sustainability_data)

DietType,Age,Gender,Rating,EnvironmentalAwareness,SustainableBrands,LocalFoodFrequency,DisposalMethods
<fct>,<dbl>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<fct>
Mostly Plant-Based,35,Female,5,5,TRUE,Often,Composting
Balanced,28,Male,4,4,TRUE,Sometimes,Recycling
Mostly Animal-Based,65,Male,1,2,FALSE,Rarely,Landfill
Mostly Plant-Based,42,Female,5,4,TRUE,Often,Recycling
Balanced,31,Non-Binary,3,3,TRUE,Sometimes,Combination
Mostly Animal-Based,58,Male,1,1,FALSE,Rarely,Landfill


In [43]:
# Releveling

# Dependent Variable: Force "Balanced" to be the baseline reference group
new_sustainability_data$DietType <- relevel(new_sustainability_data$DietType, ref = "Balanced")

# Predictors:
new_sustainability_data$Gender <- relevel(new_sustainability_data$Gender, ref = "Female")
new_sustainability_data$SustainableBrands <- relevel(new_sustainability_data$SustainableBrands, ref = "TRUE")
new_sustainability_data$LocalFoodFrequency <- relevel(new_sustainability_data$LocalFoodFrequency, ref = "Sometimes")
new_sustainability_data$DisposalMethods <- relevel(new_sustainability_data$DisposalMethods, ref = "Landfill")

## Model Fitting

In [45]:
# Fit the model
model <- multinom(DietType ~ Age + Gender + Rating + EnvironmentalAwareness + SustainableBrands + LocalFoodFrequency + DisposalMethods,
                  data = new_sustainability_data)

# View the raw coefficients and standard errors
summary(model)

# weights:  42 (26 variable)
initial  value 465.811610 
iter  10 value 313.995235
iter  20 value 294.687344
iter  30 value 294.587443
final  value 294.587399 
converged


Call:
multinom(formula = DietType ~ Age + Gender + Rating + EnvironmentalAwareness + 
    SustainableBrands + LocalFoodFrequency + DisposalMethods, 
    data = new_sustainability_data)

Coefficients:
                    (Intercept)        Age GenderMale GenderNon-Binary
Mostly Animal-Based   -2.296554 0.00107919  0.6121833        1.0181338
Mostly Plant-Based    -4.173063 0.03137655 -0.3915319        0.1220655
                        Rating EnvironmentalAwareness SustainableBrandsFALSE
Mostly Animal-Based -0.2213931              0.2023303              1.1159761
Mostly Plant-Based   0.8035697              0.1480811             -0.5987025
                    LocalFoodFrequencyAlways LocalFoodFrequencyOften
Mostly Animal-Based                2.9013995               0.8691128
Mostly Plant-Based                 0.9889842              -0.2420406
                    LocalFoodFrequencyRarely DisposalMethodsCombination
Mostly Animal-Based                2.4592167                 0.02308549
Mostl